In [ ]:
# Cell 1: Install required packages
%pip install timm torch torchvision scikit-learn matplotlib seaborn numpy pillow

: 

In [ ]:
# Cell 2: All imports
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, accuracy_score
)
from sklearn.preprocessing import label_binarize
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.decomposition import PCA

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# Cell 3: Config
TRAIN_VAL_DIR = r"C:\Users\praha\Downloads\EPICS_1\Train_Validation sets\Train_Validation sets"
TEST_DIR      = r"C:\Users\praha\Downloads\EPICS_1\Independent Test Set\Independent Test Set"

CLASS_NAMES   = ["Keratoconus", "Normal", "Suspect"]
NUM_CLASSES   = 3
IMG_SIZE      = 224
BATCH_SIZE    = 16
EPOCHS        = 20          # increased from 15
LR            = 0.00003     # lower LR for careful learning
VAL_SPLIT     = 0.2
SEED          = 42

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print("✅ Config ready — Swin-B with Domain Shift Fix")

In [ ]:
# Cell 4: Stronger augmentation to bridge domain shift
# Key idea: make training Normal images look more like test Normal images
# by applying aggressive brightness/contrast/quality variations

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(25),

    # AGGRESSIVE color/brightness augmentation
    # This forces model to recognize Normal regardless of lighting
    transforms.ColorJitter(
        brightness=0.5,    # increased heavily
        contrast=0.5,      # increased heavily
        saturation=0.3,
        hue=0.15
    ),

    # Simulate different image quality
    transforms.RandomAffine(
        degrees=15,
        translate=(0.15, 0.15),
        scale=(0.8, 1.2),   # more zoom variation
        shear=10
    ),

    transforms.RandomGrayscale(p=0.15),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.3),  # NEW
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),   # NEW — simulates blur
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2)),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("✅ Transforms ready — aggressive augmentation for domain shift")

In [ ]:
# Cell 6: Complete self-contained — builds everything from scratch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import numpy as np
import torch

# ── Settings (redefine in case kernel restarted) ───────
TRAIN_VAL_DIR = r"C:\Users\praha\Downloads\EPICS_1\Train_Validation sets\Train_Validation sets"
TEST_DIR      = r"C:\Users\praha\Downloads\EPICS_1\Independent Test Set\Independent Test Set"
CLASS_NAMES   = ["Keratoconus", "Normal", "Suspect"]
NUM_CLASSES   = 3
BATCH_SIZE    = 16
VAL_SPLIT     = 0.2
SEED          = 42
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Transforms ─────────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.3, hue=0.15),
    transforms.RandomAffine(degrees=15, translate=(0.15, 0.15), scale=(0.8, 1.2), shear=10),
    transforms.RandomGrayscale(p=0.15),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.3),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2)),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── Load full datasets ─────────────────────────────────
full_dataset_notf = datasets.ImageFolder(TRAIN_VAL_DIR, transform=val_test_transforms)
full_dataset_tf   = datasets.ImageFolder(TRAIN_VAL_DIR, transform=train_transforms)
test_dataset      = datasets.ImageFolder(TEST_DIR,      transform=val_test_transforms)

CLASS_NAMES = full_dataset_notf.classes
print(f"Classes detected : {CLASS_NAMES}")
print(f"Total train+val  : {len(full_dataset_notf)}")
print(f"Total test       : {len(test_dataset)}")

# ── Stratified split ───────────────────────────────────
targets   = [s[1] for s in full_dataset_notf.samples]
train_idx, val_idx = train_test_split(
    list(range(len(targets))),
    test_size    = VAL_SPLIT,
    stratify     = targets,
    random_state = SEED
)

train_dataset = Subset(full_dataset_tf,   train_idx)
val_dataset   = Subset(full_dataset_notf, val_idx)

# ── Class counts and weights ───────────────────────────
train_targets = [targets[i] for i in train_idx]
class_counts  = np.bincount(train_targets)
total_samples = len(train_targets)

# Equal weights since data is balanced
class_weights = torch.tensor([1.0, 1.0, 1.0], dtype=torch.float).to(DEVICE)

print("\nClass distribution (train):", dict(zip(CLASS_NAMES, class_counts)))
print("Class weights (equal)      :", dict(zip(CLASS_NAMES, [1.0, 1.0, 1.0])))

# ── DataLoaders ────────────────────────────────────────
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=False)

print(f"\nTrain  : {len(train_dataset)}")
print(f"Val    : {len(val_dataset)}")
print(f"Test   : {len(test_dataset)}")
print("✅ DataLoaders ready")


In [ ]:
# Cell 7: Image Grid — fully self contained
import random
import matplotlib.pyplot as plt
import torch
import numpy as np
from torchvision import datasets, transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
CLASS_NAMES   = ["Keratoconus", "Normal", "Suspect"]
TRAIN_VAL_DIR = r"C:\Users\praha\Downloads\EPICS_1\Train_Validation sets\Train_Validation sets"

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_dataset_notf = datasets.ImageFolder(TRAIN_VAL_DIR, transform=val_test_transforms)

def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)

def show_image_grid(dataset, class_names, n_per_class=5,
                    title="Sample Corneal Images per Class"):
    all_labels = [s[1] for s in dataset.samples]

    class_indices = {c: [] for c in range(len(class_names))}
    for idx, label in enumerate(all_labels):
        class_indices[label].append(idx)

    border_colors = ["#e74c3c", "#2ecc71", "#3498db"]
    fig, axes = plt.subplots(
        len(class_names), n_per_class,
        figsize=(n_per_class * 2.8, len(class_names) * 2.8)
    )

    for row, cls in enumerate(range(len(class_names))):
        chosen = random.sample(
            class_indices[cls],
            min(n_per_class, len(class_indices[cls]))
        )
        for col in range(n_per_class):
            ax = axes[row][col]
            if col < len(chosen):
                img_tensor, _ = dataset[chosen[col]]
                img = denormalize(img_tensor).permute(1, 2, 0).numpy()
                ax.imshow(img)
                for spine in ax.spines.values():
                    spine.set_edgecolor(border_colors[cls % 3])
                    spine.set_linewidth(3)
            ax.set_xticks([])
            ax.set_yticks([])
            if row == 0:
                ax.set_title(f"Sample {col+1}", fontsize=9, color='gray')
        axes[row][0].set_ylabel(
            class_names[cls], fontsize=12, fontweight='bold',
            rotation=90, labelpad=12,
            color=border_colors[cls % 3]
        )

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=border_colors[i], label=class_names[i])
        for i in range(len(class_names))
    ]
    fig.legend(
        handles=legend_elements, loc='lower center',
        ncol=len(class_names), fontsize=11,
        bbox_to_anchor=(0.5, -0.03),
        frameon=True
    )
    fig.suptitle(title, fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig("image_grid.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Image grid saved as image_grid.png")

show_image_grid(
    full_dataset_notf, CLASS_NAMES,
    n_per_class=5,
    title="Sample Corneal Images per Class — Swin-B"
)

In [ ]:
# Cell 8: Build Swin-B Model — fully self contained
import torch
import torch.nn as nn
import timm

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 3

model = timm.create_model(
    "swin_base_patch4_window7_224",
    pretrained     = True,
    num_classes    = NUM_CLASSES,
    drop_rate      = 0.2,
    attn_drop_rate = 0.1,
    drop_path_rate = 0.3
)

# Add dropout to head
model.head.drop = nn.Dropout(p=0.5)
model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✅ Swin Transformer Base loaded!")
print(f"   drop_rate      : 0.2")
print(f"   attn_drop_rate : 0.1")
print(f"   drop_path_rate : 0.3")
print(f"   head dropout   : 0.5")
print(f"   Total params   : {total_params:,}")
print(f"   Trainable      : {trainable_params:,}")
print(f"   Device         : {DEVICE}")

In [ ]:
# Cell 9: Complete Loss + Optimizer + Scheduler — fully self contained
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LR     = 0.00003
EPOCHS = 20

# ── Focal Loss Class Definition ────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.1):
        super().__init__()
        self.gamma           = gamma
        self.alpha           = alpha
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        num_classes     = inputs.size(1)
        confidence      = 1.0 - self.label_smoothing
        smooth_val      = self.label_smoothing / num_classes
        log_probs       = torch.nn.functional.log_softmax(inputs, dim=1)
        probs           = torch.exp(log_probs)
        targets_one_hot = torch.zeros_like(inputs).scatter_(
            1, targets.unsqueeze(1), 1
        )
        smooth_targets  = targets_one_hot * confidence + smooth_val
        pt              = (probs * targets_one_hot).sum(dim=1)
        focal_weight    = (1 - pt) ** self.gamma
        if self.alpha is not None:
            alpha_t      = self.alpha[targets]
            focal_weight = focal_weight * alpha_t
        loss = -(smooth_targets * log_probs).sum(dim=1)
        loss = focal_weight * loss
        return loss.mean()

# ── Combined Loss Class Definition ────────────────────
class CombinedLoss(nn.Module):
    def __init__(self, class_weights, gamma=2.0,
                 label_smoothing=0.15,
                 focal_weight=0.6, ce_weight=0.4):
        super().__init__()
        self.focal        = FocalLoss(
            gamma=gamma,
            label_smoothing=label_smoothing
        )
        self.ce           = nn.CrossEntropyLoss(
            weight=class_weights,
            label_smoothing=label_smoothing
        )
        self.focal_weight = focal_weight
        self.ce_weight    = ce_weight

    def forward(self, inputs, targets):
        return (self.focal_weight * self.focal(inputs, targets) +
                self.ce_weight   * self.ce(inputs, targets))

# ── Balanced weights for all 3 classes ────────────────
normal_boost = torch.tensor(
    [1.2, 1.5, 1.5],
    dtype=torch.float
).to(DEVICE)

criterion = CombinedLoss(
    class_weights   = normal_boost,
    gamma           = 1.5,
    label_smoothing = 0.1,
    focal_weight    = 0.5,
    ce_weight       = 0.5
)

# ── Optimizer ──────────────────────────────────────────
optimizer = optim.AdamW(
    model.parameters(),
    lr           = LR,
    weight_decay = 0.05,
    betas        = (0.9, 0.999),
    eps          = 1e-8
)

# ── Scheduler ──────────────────────────────────────────
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        optim.lr_scheduler.LinearLR(
            optimizer,
            start_factor = 0.1,
            end_factor   = 1.0,
            total_iters  = 3
        ),
        optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max   = EPOCHS - 3,
            eta_min = 1e-7
        )
    ],
    milestones=[3]
)

print("✅ Loss / Optimizer / Scheduler ready")
print("── Loss ──────────────────────────────────")
print("   Type            : Combined Focal + Weighted CE")
print("   Focal gamma     : 1.5")
print("   Label smoothing : 0.1")
print("   Keratoconus     : 1.2x")
print("   Normal          : 1.5x")
print("   Suspect         : 1.5x")
print("   Focal/CE split  : 50/50")
print("── Optimizer ─────────────────────────────")
print(f"  AdamW  LR={LR}  weight_decay=0.05")
print("── Scheduler ─────────────────────────────")
print("   LinearWarmup (3 epochs) → CosineAnnealing")




In [ ]:
# Cell 10: Training Loop with Mixup — fully self contained
import torch
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 20

# ── Mixup ──────────────────────────────────────────────
def mixup_data(x, y, alpha=0.3):
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1
    index = torch.randperm(x.size(0)).to(DEVICE)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# ── Epoch runner ───────────────────────────────────────
def run_epoch(model, loader, criterion,
              optimizer=None, train=True, use_mixup=False):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.set_grad_enabled(train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train and use_mixup:
                imgs, labels_a, labels_b, lam = mixup_data(imgs, labels, alpha=0.3)
                outputs = model(imgs)
                loss    = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            else:
                outputs = model(imgs)
                loss    = criterion(outputs, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            preds       = outputs.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
    return total_loss / total, correct / total

# ── Phase 1 — freeze backbone ──────────────────────────
print("PHASE 1: Training head only (epochs 1-5)...")
for name, param in model.named_parameters():
    if "head" not in name:
        param.requires_grad = False

history      = {"train_loss": [], "val_loss": [],
                "train_acc":  [], "val_acc":  []}
best_val_acc = 0.0

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | "
      f"{'Val Loss':>8} | {'Val Acc':>7} | {'Gap':>6}")
print("-" * 65)

for epoch in range(1, EPOCHS + 1):

    # Phase 2 — unfreeze after epoch 5
    if epoch == 6:
        print("\nPHASE 2: Unfreezing all layers...\n")
        for param in model.parameters():
            param.requires_grad = True

    use_mixup = (epoch > 5)

    tr_loss, tr_acc = run_epoch(
        model, train_loader, criterion,
        optimizer, train=True, use_mixup=use_mixup
    )
    vl_loss, vl_acc = run_epoch(
        model, val_loader, criterion,
        train=False, use_mixup=False
    )
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)

    gap      = tr_acc - vl_acc
    gap_flag = "✅" if gap < 0.10 else "⚠️" if gap < 0.18 else "❌"

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), "swinb_best_model.pth")

    print(f"{epoch:>6} | {tr_loss:>10.4f} | {tr_acc*100:>8.2f}% | "
          f"{vl_loss:>8.4f} | {vl_acc*100:>6.2f}% | "
          f"{gap*100:>5.1f}% {gap_flag}")

print(f"\n✅ Training complete! Best Val: {best_val_acc*100:.2f}%")
torch.save(model.state_dict(), "swinb_final_model.pth")
print("Saved → swinb_best_model.pth & swinb_final_model.pth")

In [ ]:
# Cell 11: Training Curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history["train_acc"], label="Train Acc",
             color="royalblue",  linewidth=2)
axes[0].plot(history["val_acc"],   label="Val Acc",
             color="darkorange", linewidth=2)
axes[0].set_title("Accuracy Curve — Swin-B",
                  fontsize=14, fontweight='bold')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["train_loss"], label="Train Loss",
             color="royalblue",  linewidth=2)
axes[1].plot(history["val_loss"],   label="Val Loss",
             color="darkorange", linewidth=2)
axes[1].set_title("Loss Curve — Swin-B",
                  fontsize=14, fontweight='bold')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Swin-B Training History",
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("swinb_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: swinb_training_curves.png")

In [ ]:
# Cell 12: Evaluation helper functions
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = ["Keratoconus", "Normal", "Suspect"]

def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs    = imgs.to(DEVICE)
            outputs = model(imgs)
            probs   = torch.softmax(outputs, dim=1).cpu().numpy()
            preds   = outputs.argmax(dim=1).cpu().numpy()
            all_probs.append(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds), np.vstack(all_probs)

def plot_confusion_matrix(y_true, y_pred, class_names, title="Confusion Matrix"):
    cm  = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names,
                linewidths=0.5, ax=ax,
                annot_kws={"size": 13})
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("Actual",    fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    fname = title.lower().replace(" ", "_").replace("—","") + ".png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

def plot_roc_curves(y_true, y_probs, class_names, title="ROC Curves"):
    y_bin  = label_binarize(y_true, classes=list(range(len(class_names))))
    colors = ["royalblue", "darkorange", "green"]
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, (cls, col) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
        roc_auc     = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=col, lw=2,
                label=f"{cls} (AUC = {roc_auc:.3f})")
    ax.plot([0,1],[0,1],'k--', lw=1)
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate",  fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fname = title.lower().replace(" ", "_").replace("—","") + ".png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

print("✅ Evaluation helpers ready")

In [ ]:
# Cell 13: Train Set Evaluation
from sklearn.metrics import classification_report

print("=" * 55)
print("TRAINING SET EVALUATION — Swin-B")
print("=" * 55)
tr_labels, tr_preds, tr_probs = get_predictions(model, train_loader)
print(classification_report(tr_labels, tr_preds,
      target_names=CLASS_NAMES, digits=4))
plot_confusion_matrix(tr_labels, tr_preds, CLASS_NAMES,
                      title="Confusion Matrix — Train SwinB")
plot_roc_curves(tr_labels, tr_probs, CLASS_NAMES,
                title="ROC Curves — Train SwinB")

In [ ]:
# Cell 14: Validation Set Evaluation
from sklearn.metrics import classification_report

print("=" * 55)
print("VALIDATION SET EVALUATION — Swin-B")
print("=" * 55)
vl_labels, vl_preds, vl_probs = get_predictions(model, val_loader)
print(classification_report(vl_labels, vl_preds,
      target_names=CLASS_NAMES, digits=4))
plot_confusion_matrix(vl_labels, vl_preds, CLASS_NAMES,
                      title="Confusion Matrix — Val SwinB")
plot_roc_curves(vl_labels, vl_probs, CLASS_NAMES,
                title="ROC Curves — Val SwinB")

In [ ]:
# Cell 15: Test Evaluation with TTA + Final Summary
import torch
import timm
import torch.nn as nn
import numpy as np
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import os

DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_DIR      = r"C:\Users\praha\Downloads\EPICS_1\Independent Test Set\Independent Test Set"
CLASS_NAMES   = ["Keratoconus", "Normal", "Suspect"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Load best model
if os.path.exists("swinb_best_model.pth"):
    model.load_state_dict(torch.load("swinb_best_model.pth", map_location=DEVICE))
    print("✅ Loaded: swinb_best_model.pth")
else:
    print("⚠️ Using current model weights")
model.eval()

# TTA transforms
tta_transforms = [
    transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    transforms.Compose([transforms.Resize((224,224)), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    transforms.Compose([transforms.Resize((236,236)), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    transforms.Compose([transforms.Resize((224,224)), transforms.RandomRotation(degrees=(10,10)), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    transforms.Compose([transforms.Resize((224,224)), transforms.RandomRotation(degrees=(-10,-10)), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
]

def get_tta_predictions(model, test_dir, tta_transforms):
    model.eval()
    all_probs = []
    for tfm in tta_transforms:
        dataset    = datasets.ImageFolder(test_dir, transform=tfm)
        loader     = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=0)
        probs_this = []
        with torch.no_grad():
            for imgs, _ in loader:
                imgs    = imgs.to(DEVICE)
                outputs = model(imgs)
                probs   = torch.softmax(outputs, dim=1).cpu().numpy()
                probs_this.append(probs)
        all_probs.append(np.vstack(probs_this))
    avg_probs    = np.mean(all_probs, axis=0)
    preds        = np.argmax(avg_probs, axis=1)
    base_dataset = datasets.ImageFolder(test_dir, transform=tta_transforms[0])
    labels       = [s[1] for s in base_dataset.samples]
    return np.array(labels), preds, avg_probs

print("=" * 55)
print("TEST SET EVALUATION (TTA) — Swin-B")
print("=" * 55)
te_labels, te_preds, te_probs = get_tta_predictions(
    model, TEST_DIR, tta_transforms
)
print(classification_report(te_labels, te_preds,
      target_names=CLASS_NAMES, digits=4))
plot_confusion_matrix(te_labels, te_preds, CLASS_NAMES,
                      title="Confusion Matrix — Test SwinB")
plot_roc_curves(te_labels, te_probs, CLASS_NAMES,
                title="ROC Curves — Test SwinB")

# Final Summary
print("\n" + "=" * 50)
print("     FINAL ACCURACY SUMMARY — Swin-B")
print("=" * 50)
print(f"  Train Accuracy      : {accuracy_score(tr_labels, tr_preds)*100:.2f}%")
print(f"  Validation Accuracy : {accuracy_score(vl_labels, vl_preds)*100:.2f}%")
print(f"  Test Accuracy (TTA) : {accuracy_score(te_labels, te_preds)*100:.2f}%")
print("=" * 50)
print("\n  Per-Class Test Accuracy:")
print("-" * 35)
cm_final = confusion_matrix(te_labels, te_preds)
for cls, acc in zip(CLASS_NAMES, cm_final.diagonal() / cm_final.sum(axis=1)):
    print(f"  {cls:<15} : {acc*100:.2f}%")
print("-" * 35)
print("=" * 50)

In [ ]:
# Cell 16: Decision Tree Feature Extraction — fully self contained
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = ["Keratoconus", "Normal", "Suspect"]
SEED = 42

class SwinFeatureExtractor(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x):
        x = self.backbone.forward_features(x)
        if x.dim() == 3:
            x = x.mean(dim=1)
        elif x.dim() == 4:
            x = x.mean(dim=[1, 2])
        return x

feature_extractor = SwinFeatureExtractor(model).to(DEVICE)
feature_extractor.eval()

def extract_features(loader):
    feats, lbls = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            f    = feature_extractor(imgs).cpu().numpy()
            feats.append(f)
            lbls.extend(labels.numpy())
    return np.vstack(feats), np.array(lbls)

print("Extracting train features...")
X_train_feats, y_train_feats = extract_features(train_loader)
print("Extracting test features...")
X_test_feats,  y_test_feats  = extract_features(test_loader)
print(f"Feature shape — Train: {X_train_feats.shape} | Test: {X_test_feats.shape}")

# PCA
pca         = PCA(n_components=50, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_feats)
X_test_pca  = pca.transform(X_test_feats)
print(f"After PCA — Train: {X_train_pca.shape} | Test: {X_test_pca.shape}")

# Decision Tree
dt = DecisionTreeClassifier(
    max_depth=5, random_state=SEED, class_weight='balanced'
)
dt.fit(X_train_pca, y_train_feats)
dt_preds = dt.predict(X_test_pca)

print("\nDecision Tree on Swin-B Features:")
print(classification_report(y_test_feats, dt_preds,
      target_names=CLASS_NAMES, digits=4))

In [ ]:
# Cell 17: Decision Tree Visualization
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(28, 12))
plot_tree(
    dt,
    feature_names = [f"PCA_{i}" for i in range(50)],
    class_names   = CLASS_NAMES,
    filled        = True,
    rounded       = True,
    fontsize      = 9,
    ax            = ax,
    impurity      = True,
    proportion    = False
)
plt.title(
    "Decision Tree on Swin-B Features (PCA-50, depth=5)",
    fontsize=15, fontweight='bold', pad=20
)
plt.tight_layout()
plt.savefig("swinb_decision_tree.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: swinb_decision_tree.png")

In [ ]:
# Cell 18: Decision Tree Confusion Matrix
plot_confusion_matrix(
    y_test_feats, dt_preds, CLASS_NAMES,
    title="Confusion Matrix — Decision Tree SwinB"
)

In [ ]:
# Cell 19: AUC Summary Table
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

CLASS_NAMES = ["Keratoconus", "Normal", "Suspect"]

def compute_aucs(y_true, y_probs, class_names):
    y_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    return {
        cls: round(auc(*roc_curve(y_bin[:, i], y_probs[:, i])[:2]), 4)
        for i, cls in enumerate(class_names)
    }

tr_aucs = compute_aucs(tr_labels, tr_probs, CLASS_NAMES)
vl_aucs = compute_aucs(vl_labels, vl_probs, CLASS_NAMES)
te_aucs = compute_aucs(te_labels, te_probs, CLASS_NAMES)

print("\n" + "=" * 58)
print("   AUC SUMMARY — Swin Transformer Base")
print("=" * 58)
print(f"{'Class':<20} | {'Train AUC':>9} | {'Val AUC':>7} | {'Test AUC':>8}")
print("-" * 58)
for cls in CLASS_NAMES:
    print(f"{cls:<20} | {tr_aucs[cls]:>9} | {vl_aucs[cls]:>7} | {te_aucs[cls]:>8}")
print("=" * 58)

In [ ]:
# Cell 20: Final Summary
print("\n✅ Swin-B Corneal Disease Detection — All outputs:")
outputs = [
    "swinb_best_model.pth           — Best model (best val epoch)",
    "swinb_final_model.pth          — Final epoch model",
    "image_grid.png                 — Sample images per class",
    "swinb_training_curves.png      — Accuracy & Loss curves",
    "confusion_matrix_train.png     — Train confusion matrix",
    "confusion_matrix_val.png       — Val confusion matrix",
    "confusion_matrix_test.png      — Test confusion matrix",
    "roc_curves_train.png           — Train ROC curves",
    "roc_curves_val.png             — Val ROC curves",
    "roc_curves_test.png            — Test ROC curves (TTA)",
    "swinb_decision_tree.png        — Decision Tree visualization",
    "confusion_matrix_decision_tree — DT confusion matrix",
]
for o in outputs:
    print(f"  📁 {o}")

print("\n🎉 Project complete — Swin-B Corneal Disease Detection!")
